# Módulo 18: KDD y CRISP-DM — Guía del ejercicio

En este ejercicio vas a recorrer el ciclo CRISP-DM en un problema de abandono de clientes: primero relacionas sus fases con el caso y las comparas con KDD, y después las aplicas en Python, desde la exploración de los datos hasta el uso del modelo para decidir a qué clientes contactar.

- Si necesitas repasar los conceptos, consulta la lección [m18-kdd-crisp-dm-leccion.md](m18-kdd-crisp-dm-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m18-kdd-crisp-dm-solucion.ipynb](m18-kdd-crisp-dm-solucion.ipynb).

## Contexto

**CineNube** es un servicio ficticio de películas y series por suscripción. En el último año, cada vez más suscriptores cancelan, y la dirección quiere reducir el abandono antes de invertir más en captar clientes nuevos, que cuesta varias veces más que retener a uno.

El equipo de retención puede llamar cada mes al **10 % de los suscriptores activos** para ofrecerles un descuento o un cambio de plan. Hoy elige a quién llamar casi al azar. Te piden un modelo que ordene a los suscriptores por riesgo de abandono, para que esas llamadas lleguen a quienes de verdad están a punto de irse, y que el proyecto se organice con una metodología reconocida.

## Datos

La celda inicial genera, con una semilla fija, dos DataFrames:

- `clientes`: 4000 suscriptores de un periodo pasado, con la columna `abandono` que indica si cancelaron en los tres meses siguientes. Sirve para entrenar y evaluar el modelo.
- `cartera`: 1000 suscriptores activos hoy, con las mismas columnas salvo `abandono`, porque todavía no se sabe quién se irá. Es la cartera que hay que puntuar.

| Columna | Descripción |
| --- | --- |
| `cliente_id` | Identificador del suscriptor |
| `tipo_contrato` | `mensual` (se puede cancelar cualquier mes) o `anual` |
| `metodo_pago` | `tarjeta`, `domiciliado` (cargo automático a la cuenta bancaria) o `manual` (el cliente paga cada mes) |
| `antiguedad_meses` | Meses como suscriptor |
| `cargo_mensual` | Importe mensual en dólares, incluidos los complementos contratados |
| `horas_uso_semana` | Horas de visualización por semana en el último mes. Tiene valores faltantes: la aplicación no registró el uso de algunos clientes |
| `llamadas_soporte` | Contactos con soporte técnico en los últimos tres meses |
| `abandono` | 1 si canceló en los tres meses siguientes, 0 si no (solo en `clientes`) |

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta (`uv run jupyter lab` desde la raíz del repositorio).
- Las tareas 1 a 3 son preguntas escritas: respóndelas en la celda de texto que las sigue, aplicando los conceptos de la lección a CineNube.
- Las tareas 4 a 7 recorren en código las fases de CRISP-DM que se trabajan con los datos. Escribe tu código en las celdas vacías y acompaña los resultados con una interpretación breve. Puedes añadir más celdas si lo necesitas.
- Usa `random_state=42` donde haya azar y no modifiques la celda inicial, para que tus resultados sean comparables con la solución.

La celda siguiente importa las librerías y genera los datos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)


def generar(n):
    """Suscriptores sintéticos; el abandono sigue un modelo logístico de sus variables."""
    contrato = rng.choice(["mensual", "anual"], size=n, p=[0.6, 0.4])
    antiguedad = np.where(contrato == "anual", rng.integers(6, 61, n), rng.integers(1, 49, n))
    pago = rng.choice(["tarjeta", "domiciliado", "manual"], size=n, p=[0.45, 0.35, 0.20])
    cargo = rng.choice([6.99, 10.99, 15.99], size=n, p=[0.35, 0.45, 0.20]) + 2.5 * rng.poisson(0.4, n)
    uso = np.round(rng.gamma(2.5, 2.4, n), 1)
    llamadas = rng.poisson(0.5 + 0.7 * (uso < 3), n)
    z = (-0.9 + 1.4 * (contrato == "mensual") + 0.6 * (pago == "manual") - 0.2 * (pago == "domiciliado")
         - 0.045 * antiguedad + 0.10 * (cargo - 10) + 0.5 * llamadas - 0.2 * uso)
    abandono = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    uso_registrado = np.where(rng.random(n) < 0.03, np.nan, uso)   # ~3 % sin registro
    return pd.DataFrame({
        "cliente_id": np.arange(1, n + 1),
        "tipo_contrato": contrato,
        "metodo_pago": pago,
        "antiguedad_meses": antiguedad,
        "cargo_mensual": cargo.round(2),
        "horas_uso_semana": uso_registrado,
        "llamadas_soporte": llamadas,
        "abandono": abandono,
    })


clientes = generar(4000)
cartera = generar(1000).drop(columns="abandono")
cartera["cliente_id"] += 10000

print("clientes:", clientes.shape, " cartera:", cartera.shape)
clientes.head()

## Tarea 1. Las seis fases de CRISP-DM en CineNube

Para cada una de las seis fases de CRISP-DM (comprensión del negocio, comprensión de los datos, preparación de los datos, modelado, evaluación y despliegue), escribe qué se haría concretamente en CineNube: preguntas que se responden, actividades y entregable de la fase. En la comprensión del negocio, incluye un objetivo medible y el criterio con el que se juzgará el éxito del proyecto.

*Escribe aquí tu respuesta.*

## Tarea 2. KDD frente a CRISP-DM

1. Enumera las etapas del proceso KDD y relaciona cada una con la fase o las fases de CRISP-DM que le corresponden.
2. Explica en qué se diferencian las dos metodologías: qué fases tiene CRISP-DM que KDD no desarrolla, y qué pone KDD en el centro.

*Escribe aquí tu respuesta.*

## Tarea 3. ¿Qué metodología conviene?

1. Recomienda KDD o CRISP-DM para el proyecto de CineNube y justifica tu elección con al menos tres argumentos ligados al caso.
2. Indica una desventaja de la metodología elegida que podría afectar a este proyecto y cómo la mitigarías.

*Escribe aquí tu respuesta.*

## Tarea 4. Comprensión de los datos

1. Revisa la estructura de `clientes`: dimensiones, tipos de datos y valores faltantes por columna.
2. Calcula la **tasa de abandono** global, en porcentaje.
3. Calcula la tasa de abandono por `tipo_contrato`, por `metodo_pago` y por número de `llamadas_soporte` (agrupa los valores de 4 o más en una sola categoría).
4. Compara la media de `antiguedad_meses`, `cargo_mensual` y `horas_uso_semana` entre quienes abandonan y quienes se quedan.
5. Haz una figura con dos paneles: la tasa de abandono por tipo de contrato y la tasa de abandono por número de llamadas a soporte.
6. Resume en tres hallazgos qué perfiles abandonan más.

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 5. Preparación de los datos

1. Separa las variables explicativas `X` (todas salvo `cliente_id` y `abandono`) y el objetivo `y`.
2. Convierte `tipo_contrato` y `metodo_pago` en variables dummy con `pd.get_dummies(..., drop_first=True, dtype=int)`. ¿Qué categoría queda como referencia en cada caso?
3. Divide los datos en entrenamiento (75 %) y prueba (25 %) con `train_test_split`, estratificando por `y` y con `random_state=42`. Comprueba que la tasa de abandono es casi igual en los dos conjuntos.
4. Explica por qué conviene estratificar y por qué los valores faltantes de `horas_uso_semana` se imputan **después** de dividir, dentro del modelo, y no antes con todos los datos.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 6. Modelado y evaluación

1. Construye un `Pipeline` con tres pasos: `SimpleImputer(strategy="median")`, `StandardScaler()` y `LogisticRegression(max_iter=1000)`. Entrénalo con el conjunto de entrenamiento.
2. En el conjunto de prueba, calcula la probabilidad de abandono de cada cliente con `predict_proba` y las predicciones con el umbral por defecto de 0.5.
3. Muestra la matriz de confusión y calcula el **recall** y la **precisión** de la clase 1, y el **ROC AUC** con las probabilidades.
4. Muestra los coeficientes de la regresión logística, ordenados. Como las variables están estandarizadas, puedes comparar su magnitud.
5. Interpreta: ¿qué variables aumentan el riesgo y cuáles lo reducen? ¿Es suficiente el recall con el umbral 0.5 para el equipo de retención?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 7. Despliegue: a quién llamar este mes

El presupuesto alcanza para llamar al 10 % de los suscriptores activos.

1. En el conjunto de prueba, ordena a los clientes por probabilidad de abandono y quédate con el 10 % de mayor riesgo (el umbral es el percentil 90 de las probabilidades). Calcula qué proporción de todos los que abandonan cae en ese grupo (**tasa de captura**) y qué proporción del grupo abandona de verdad (**precisión**). Compara con llamar a un 10 % elegido al azar.
2. Dibuja la **curva de captura**: porcentaje de clientes contactados, ordenados de mayor a menor riesgo (eje x), frente al porcentaje de abandonos capturados (eje y), con la diagonal que corresponde a contactar al azar. Marca el punto del 10 %.
3. Puntúa la `cartera` con el modelo entrenado, selecciona al 10 % con mayor riesgo y muestra cuántos clientes son, el umbral de probabilidad que resulta y los diez primeros de la lista.
4. Interpreta: ¿qué le dirías a la dirección sobre el beneficio esperado de usar el modelo? ¿Qué habría que monitorear una vez desplegado?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Antes de terminar

- [ ] Las tareas 1 a 3 aplican las fases y las metodologías a CineNube, no las describen en abstracto.
- [ ] El notebook se ejecuta de principio a fin sin errores.
- [ ] La división en entrenamiento y prueba está estratificada y usa `random_state=42`.
- [ ] La imputación y el escalado están dentro del `Pipeline`.
- [ ] La evaluación incluye matriz de confusión, recall y ROC AUC, y el despliegue usa el presupuesto del 10 %.
- [ ] Cada resultado va acompañado de una interpretación breve.

Cuando termines, compara tu trabajo con la solución: [m18-kdd-crisp-dm-solucion.ipynb](m18-kdd-crisp-dm-solucion.ipynb).